# Week 2 Day 2 hard challenge: reply to prospects with an SDR agent

The goal is to receive a prospect's email reply through SendGrid, give the conversation to an SDR agent, and send one useful response.

The live SendGrid round trip has not been tested because the original trial account expired. The notebook therefore separates the reply logic from the external services:

- A local test checks the webhook logic without calling OpenAI or SendGrid.
- An optional test calls the SDR agent and prints the outgoing message when SendGrid is not configured.
- The FastAPI endpoint and SendGrid Mail Send request are included for live use.

All addresses and domains are placeholders.

## How it works

1. The first cold email uses a reply address on a SendGrid Inbound Parse subdomain.
2. The prospect replies to that address.
3. SendGrid posts the parsed message to the FastAPI endpoint as `multipart/form-data`.
4. The handler checks the sender, message ID, reply text, and conversation limit.
5. The SDR agent writes a response using the saved conversation.
6. The reply is sent through the SendGrid Mail Send API, or printed when SendGrid is not configured.

Before sending the first cold email, call `start_conversation(...)` so the reply handler knows the original email and does not answer unrelated inbound mail.

Live setup requires:

- A dedicated subdomain with an MX record pointing to `mx.sendgrid.net`.
- A SendGrid Inbound Parse setting for that subdomain and a public HTTPS endpoint.
- A verified SendGrid sender.
- Environment variables for `SENDGRID_API_KEY`, `SENDGRID_FROM_EMAIL`, `SENDGRID_REPLY_TO`, and `INBOUND_SECRET`.

References: [Inbound Parse setup](https://www.twilio.com/docs/sendgrid/for-developers/parsing-email/setting-up-the-inbound-parse-webhook), [Mail Send API](https://www.twilio.com/docs/sendgrid/api-reference/mail-send/mail-send), and [securing Inbound Parse webhooks](https://www.twilio.com/docs/sendgrid/for-developers/parsing-email/securing-your-parse-webhooks).

In [ ]:
import os
import re
import secrets
from email.parser import Parser
from email.utils import parseaddr

import requests
from dotenv import load_dotenv
from agents import Agent, Runner, trace

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
MAX_AUTOMATED_REPLIES = 3

SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY")
SENDGRID_FROM_EMAIL = os.getenv("SENDGRID_FROM_EMAIL")
SENDGRID_REPLY_TO = os.getenv("SENDGRID_REPLY_TO")
INBOUND_SECRET = os.getenv("INBOUND_SECRET")

SENDGRID_MAIL_URL = "https://api.sendgrid.com/v3/mail/send"

# Demo storage. Use a database for a deployed service.
conversations: dict[tuple[str, str], list[dict[str, str]]] = {}
processed_message_ids: set[str] = set()

In [ ]:
sdr_reply_agent = Agent(
    name="SDR Reply Agent",
    instructions=(
        "You are an SDR for ComplAI, a tool for SOC2 compliance. "
        "Read the email conversation and write a brief, helpful reply to the prospect. "
        "Answer the question using only information supported by the conversation and the product description. "
        "Do not invent prices, delivery dates, guarantees, or product features. "
        "Suggest a short call when it would help, but do not pressure the prospect. "
        "If a person needs to answer the question, say that you will ask a colleague to follow up. "
        "Return only the email body."
    ),
    model=MODEL_NAME,
)

In [ ]:
def parsed_headers(raw_headers: str):
    return Parser().parsestr(raw_headers or "")

def sender_address(value: str) -> str:
    return parseaddr(value or "")[1].lower()

def inbound_message_id(raw_headers: str) -> str:
    return parsed_headers(raw_headers).get("Message-ID", "").strip()

def is_auto_reply(sender: str, raw_headers: str) -> bool:
    """Skip no-reply addresses, automatic replies, and bulk mail."""
    sender = sender.lower()
    headers = parsed_headers(raw_headers)

    if "no-reply" in sender or "noreply" in sender:
        return True

    auto_submitted = headers.get("Auto-Submitted", "").strip().lower()
    if auto_submitted and auto_submitted != "no":
        return True

    precedence = headers.get("Precedence", "").strip().lower()
    return precedence in {"bulk", "junk", "list"}

def strip_quoted(text: str) -> str:
    """Keep the new reply and drop common quoted-history markers."""
    lines = []
    for line in (text or "").splitlines():
        if (
            line.startswith(">")
            or re.match(r"On .+ wrote:", line)
            or line.strip().lower() == "-----original message-----"
        ):
            break
        lines.append(line)
    return "\n".join(lines).strip()

def normalised_subject(subject: str) -> str:
    return re.sub(r"^(re:\s*)+", "", (subject or "").strip(), flags=re.I).lower()

def thread_key(email: str, subject: str) -> tuple[str, str]:
    return sender_address(email), normalised_subject(subject)

def reply_subject(subject: str) -> str:
    subject = (subject or "").strip()
    return subject if subject.lower().startswith("re:") else f"Re: {subject}"

def transcript(messages: list[dict[str, str]]) -> str:
    return "\n\n".join(f"{message['role']}: {message['text']}" for message in messages)

def start_conversation(recipient: str, subject: str, cold_email: str) -> None:
    """Save the first outbound email before waiting for a reply."""
    conversations[thread_key(recipient, subject)] = [
        {"role": "SDR", "text": cold_email, "source": "initial"}
    ]

In [ ]:
def send_reply(
    to: str,
    subject: str,
    body: str,
    in_reply_to: str = "",
) -> str:
    """Send through SendGrid, or print a preview when credentials are missing."""
    if not (SENDGRID_API_KEY and SENDGRID_FROM_EMAIL):
        print(
            f"Email preview\n"
            f"To: {to}\n"
            f"Subject: {subject}\n"
            f"In-Reply-To: {in_reply_to or '[not supplied]'}\n\n"
            f"{body}"
        )
        return "previewed"

    personalisation = {"to": [{"email": to}]}
    if in_reply_to:
        personalisation["headers"] = {
            "In-Reply-To": in_reply_to,
            "References": in_reply_to,
        }

    message = {
        "personalizations": [personalisation],
        "from": {"email": SENDGRID_FROM_EMAIL},
        "subject": subject,
        "content": [{"type": "text/plain", "value": body}],
    }

    if SENDGRID_REPLY_TO:
        message["reply_to"] = {"email": SENDGRID_REPLY_TO}

    response = requests.post(
        SENDGRID_MAIL_URL,
        headers={
            "Authorization": f"Bearer {SENDGRID_API_KEY}",
            "Content-Type": "application/json",
        },
        json=message,
        timeout=30,
    )
    response.raise_for_status()
    return "sent"

In [ ]:
async def write_sdr_reply(conversation_text: str) -> str:
    with trace("SDR reply"):
        result = await Runner.run(sdr_reply_agent, conversation_text)
    return result.final_output.strip()

async def handle_inbound_reply(
    payload: dict,
    reply_writer=write_sdr_reply,
    reply_sender=send_reply,
) -> dict:
    """Validate one parsed inbound email and send at most one reply."""
    sender = sender_address(payload.get("from", ""))
    subject = (payload.get("subject") or "").strip()
    raw_headers = payload.get("headers", "")
    message_id = inbound_message_id(raw_headers)

    if not sender:
        return {"status": "ignored", "reason": "missing sender"}

    if is_auto_reply(sender, raw_headers):
        return {"status": "ignored", "reason": "automatic message"}

    if message_id and message_id in processed_message_ids:
        return {"status": "ignored", "reason": "duplicate message"}

    new_text = strip_quoted(payload.get("text", ""))
    if not new_text:
        return {"status": "ignored", "reason": "empty reply"}

    key = thread_key(sender, subject)
    if key not in conversations:
        return {"status": "ignored", "reason": "unknown conversation"}

    conversation = conversations[key]
    automated_replies = sum(
        1 for message in conversation if message.get("source") == "automated"
    )
    if automated_replies >= MAX_AUTOMATED_REPLIES:
        return {"status": "ignored", "reason": "reply limit reached"}

    prospect_message = {"role": "Prospect", "text": new_text, "source": "inbound"}
    conversation_for_agent = [*conversation, prospect_message]
    response = await reply_writer(transcript(conversation_for_agent))

    if not response.strip():
        return {"status": "ignored", "reason": "empty agent response"}

    delivery = reply_sender(
        sender,
        reply_subject(subject),
        response,
        in_reply_to=message_id,
    )

    conversation.extend(
        [
            prospect_message,
            {"role": "SDR", "text": response, "source": "automated"},
        ]
    )
    if message_id:
        processed_message_ids.add(message_id)

    return {
        "status": delivery,
        "to": sender,
        "automated_reply": automated_replies + 1,
    }

## Local test without external calls

This test seeds the original cold email, passes in a saved Inbound Parse payload, and replaces both external calls with small local functions. It checks the conversation flow and duplicate handling without using an OpenAI key or SendGrid account.

In [ ]:
conversations.clear()
processed_message_ids.clear()

subject = "Streamline your SOC2 compliance"
prospect = "alex@example.com"

start_conversation(
    recipient=prospect,
    subject=subject,
    cold_email="Hi Alex, ComplAI helps teams prepare for SOC2 audits. Would a short overview be useful?",
)

sample_inbound = {
    "from": "Alex Example <alex@example.com>",
    "to": "sdr@reply.example.com",
    "subject": f"Re: {subject}",
    "text": (
        "Thanks for reaching out. How long does onboarding usually take?\n\n"
        "On Mon, the SDR wrote:\n"
        "> Hi Alex, ComplAI helps teams prepare for SOC2 audits."
    ),
    "headers": (
        "Message-ID: <reply-001@example.com>\n"
        "Auto-Submitted: no\n"
        "Received: from mail.example.com"
    ),
}

async def fixed_reply_writer(conversation_text: str) -> str:
    assert "SDR:" in conversation_text
    assert "Prospect:" in conversation_text
    return "Thanks for asking. Onboarding depends on your current setup. I can ask a colleague to give you a more specific estimate."

sent_messages = []

def record_reply(to: str, subject: str, body: str, in_reply_to: str = "") -> str:
    sent_messages.append(
        {
            "to": to,
            "subject": subject,
            "body": body,
            "in_reply_to": in_reply_to,
        }
    )
    return "recorded"

test_result = await handle_inbound_reply(
    sample_inbound,
    reply_writer=fixed_reply_writer,
    reply_sender=record_reply,
)
duplicate_result = await handle_inbound_reply(
    sample_inbound,
    reply_writer=fixed_reply_writer,
    reply_sender=record_reply,
)

assert test_result["status"] == "recorded"
assert duplicate_result == {"status": "ignored", "reason": "duplicate message"}
assert len(sent_messages) == 1
assert sent_messages[0]["in_reply_to"] == "<reply-001@example.com>"

print(test_result)
print(duplicate_result)
print(sent_messages[0])

The next cell calls the OpenAI model. When SendGrid credentials are absent, `send_reply` prints the response instead of sending it.

A separate message ID is used so this test does not look like a duplicate of the local test.

In [ ]:
agent_sample = {
    **sample_inbound,
    "text": (
        "Thanks for the note. Can your team help us understand what evidence we need for an audit?\n\n"
        "On Mon, the SDR wrote:\n"
        "> Hi Alex, ComplAI helps teams prepare for SOC2 audits."
    ),
    "headers": (
        "Message-ID: <reply-002@example.com>\n"
        "Auto-Submitted: no\n"
        "Received: from mail.example.com"
    ),
}

agent_result = await handle_inbound_reply(agent_sample)
print(agent_result)